In [ ]:
import pandas as pd


# Load the CSV files directly into DataFrames
train_df = pd.read_csv(r'data/sent_train.csv')
test_df = pd.read_csv(r'data/sent_valid (1).csv')

# Print the first 5 rows
print(train_df.head())
print(test_df.head())


In [ ]:
train_df.isnull().sum()

In [ ]:
test_df.isnull().sum()

In [ ]:
train_df['label'].value_counts()

In [ ]:
test_df.isnull().sum()

In [ ]:
test_df['label'].value_counts()

In [ ]:
#label map
label_map = {
    0: "Bearish",
    1: "Bullish",
    2: "Neutral"
}

train_df["sentiment"] = train_df["label"].map(label_map)
test_df["sentiment"] = test_df["label"].map(label_map)

In [ ]:
train_df

In [ ]:
import matplotlib.pyplot as plt

train_df["sentiment"].value_counts().plot(kind="bar",color=['red','green','yellow'])

plt.title("Financial Tweet Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Tweets")
plt.xticks(rotation=0)
plt.show()

In [ ]:
#duplicate chack

train_df.duplicated().sum()


In [ ]:
test_df.duplicated().sum()

In [ ]:
# text cleaning
import re
def clean_text(text):
    text = str(text)

    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\$(\w+)", r"\1", text)
    text = re.sub(r"[^a-zA-Z0-9.%\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [ ]:
train_df

In [ ]:
print("Original:")
print(train_df["text"].iloc[0])

print("\nCleaned:")
print(train_df["clean_text"].iloc[0])

In [ ]:
train_df = train_df[
    train_df["clean_text"].str.strip() != ""
].copy()

test_df = test_df[
    test_df["clean_text"].str.strip() != ""
].copy()

In [ ]:
import os

# Create folder
os.makedirs("data_clean", exist_ok=True)
# Save training data
train_df.to_csv(
    r"data_clean/train_clean.csv",
    index=False
)
# Save test data
test_df.to_csv(
    r"data_clean/test_clean.csv",
    index=False
)

print("Clean data saved successfully!")

In [ ]:
print(os.listdir("data_clean"))

In [ ]:
# load clean data

In [ ]:
import pandas as pd

# Load cleaned training data
train_df = pd.read_csv("data_clean/train_clean.csv")

# Load cleaned validation data
test_df = pd.read_csv("data_clean/test_clean.csv")

print("Training data shape:", train_df.shape)
print("Validation data shape:", test_df.shape)

In [ ]:
print(train_df.head())
print(test_df.head())

In [ ]:
print(train_df.columns)

In [ ]:
print(train_df["label"].unique())
print(test_df["label"].unique())

In [ ]:
#Prepare X and y
X_train = train_df["clean_text"].values
y_train = train_df["label"].values

X_test = test_df["clean_text"].values
y_test = test_df["label"].values

print("Training samples:", len(X_train))
print("Validation samples:", len(X_test))

## Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 100

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [ ]:
#Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


In [ ]:
#padding
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_valid_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Train:", X_train_pad.shape)
print("Validation:", X_valid_pad.shape)

In [ ]:
import os

os.makedirs("models", exist_ok=True)

print("Model folder ready:")
print(os.listdir("/content"))

## MODEL 1 — Simple RNN


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

rnn_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax")
])

rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
rnn_model.summary()

# train 
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_rnn = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_valid_pad, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

# evaulate
from sklearn.metrics import accuracy_score, f1_score, classification_report

rnn_probs = rnn_model.predict(X_valid_pad)

rnn_pred = rnn_probs.argmax(axis=1)

rnn_accuracy = accuracy_score(y_test, rnn_pred)

rnn_f1 = f1_score(
    y_test,
    rnn_pred,
    average="macro"
)

print("RNN Accuracy:", rnn_accuracy)
print("RNN Macro F1:", rnn_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rnn_pred,
        target_names=["Bearish", "Bullish", "Neutral"]
    )
)

#Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, rnn_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Bearish", "Bullish", "Neutral"],
    yticklabels=["Bearish", "Bullish", "Neutral"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Simple RNN - Confusion Matrix")

plt.show()

# save

rnn_model.save("models/rnn_model.keras")

print("✅ RNN model saved successfully")

## MODEL 2 — LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

#train
history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_valid_pad, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

# evaulate
lstm_probs = lstm_model.predict(X_valid_pad)

lstm_pred = lstm_probs.argmax(axis=1)

lstm_accuracy = accuracy_score(y_test, lstm_pred)

lstm_f1 = f1_score(
    y_test,
    lstm_pred,
    average="macro"
)

print("LSTM Accuracy:", lstm_accuracy)
print("LSTM Macro F1:", lstm_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        lstm_pred,
        target_names=["Bearish", "Bullish", "Neutral"]
    )
)

# confusion matrix
cm = confusion_matrix(y_test, lstm_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Bearish", "Bullish", "Neutral"],
    yticklabels=["Bearish", "Bullish", "Neutral"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LSTM - Confusion Matrix")

plt.show()

 # save
lstm_model.save("models/lstm_model.keras")

print("✅ LSTM model saved successfully")

## MODEL 3 — GRU

In [ ]:
from tensorflow.keras.layers import GRU

gru_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(64),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

gru_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

history_gru = gru_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_valid_pad, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)


gru_probs = gru_model.predict(X_valid_pad)

gru_pred = gru_probs.argmax(axis=1)

gru_accuracy = accuracy_score(y_test, gru_pred)

gru_f1 = f1_score(
    y_test,
    gru_pred,
    average="macro"
)

print("GRU Accuracy:", gru_accuracy)
print("GRU Macro F1:", gru_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        gru_pred,
        target_names=["Bearish", "Bullish", "Neutral"]
    )
)

cm = confusion_matrix(y_test, gru_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Bearish", "Bullish", "Neutral"],
    yticklabels=["Bearish", "Bullish", "Neutral"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("GRU - Confusion Matrix")

plt.show()

gru_model.save("models/gru_model.keras")

print("✅ GRU model saved successfully")

## MODEL 4 — BiLSTM

In [ ]:
from tensorflow.keras.layers import Bidirectional, LSTM

bilstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

bilstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()


# train 
history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_valid_pad, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

# evaulate
bilstm_probs = bilstm_model.predict(X_valid_pad)

bilstm_pred = bilstm_probs.argmax(axis=1)

bilstm_accuracy = accuracy_score(
    y_test,
    bilstm_pred
)

bilstm_f1 = f1_score(
    y_test,
    bilstm_pred,
    average="macro"
)

print("BiLSTM Accuracy:", bilstm_accuracy)
print("BiLSTM Macro F1:", bilstm_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        bilstm_pred,
        target_names=["Bearish", "Bullish", "Neutral"]
    )
)

#matrix
cm = confusion_matrix(y_test, bilstm_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Bearish", "Bullish", "Neutral"],
    yticklabels=["Bearish", "Bullish", "Neutral"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BiLSTM - Confusion Matrix")

plt.show()

# save 

bilstm_model.save("models/bilstm_model.keras")

print("✅ BiLSTM model saved successfully")

## Compare all 4 models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Simple RNN",
        "LSTM",
        "GRU",
        "BiLSTM"
    ],
    
    "Accuracy": [
        rnn_accuracy,
        lstm_accuracy,
        gru_accuracy,
        bilstm_accuracy
    ],
    
    "Macro F1": [
        rnn_f1,
        lstm_f1,
        gru_f1,
        bilstm_f1
    ]
})

results

results.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

results.plot(
    x="Model",
    y=["Accuracy", "Macro F1"],
    kind="bar",
    figsize=(10, 6)
)

plt.title("Financial Sentiment Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

## ## Check saved models

In [ ]:
import os

print(os.listdir("models"))

In [ ]:
results.to_csv(
    "models/model_comparison.csv",
    index=False
)

print("✅ Comparison results saved")

In [ ]:
results

In [ ]:
import pickle
import os

os.makedirs("models", exist_ok=True)

with open("models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Tokenizer saved successfully")

In [ ]:
os.listdir("models")